<table>
  <tr>
    <td><div align="left"><font size="30">Bouncing ball simulation</font></div></td>
    <td><img src="https://raw.githubusercontent.com/petercorke/bdsim/master/figs/BDSimLogo_NoBackgnd%402x.png" width="300"></td>
  </tr>
</table>

(c) Peter Corke 2026

In [ ]:
# Install bdsim.  In JupyterLite, piplite checks the local wheel
# (built with 'make wheel') before falling back to PyPI.
# In a regular Jupyter server this is a no-op (piplite is not available).
try:
    import piplite
    await piplite.install('bdsim')
except ImportError:
    pass  # already installed (classic Jupyter)


In this notebook will build a model of a simple dynamic system, a bouncy ball dropped from a certain height, and simulate it.

Basic knowledge of `bdsim` from the "Getting started" notebook is assumed.

We start with the standard boilerplate: import the package, establish the simulation environment, create an empty block diagram to work with.

In [ ]:
import bdsim
sim = bdsim.BDSim()
bd = sim.blockdiagram()

For our physical system we need to define some constants. We will assume that height is positive in the upwards direction.

In [ ]:
g = -9.81  # gravity m/s2
e = 0.8  # coefficient of restitution
h0 = 10  # initial height m
v0 = 0  # initial velocity m/s

Our model is pretty straightforward, but we have to be thinking in block diagram terms. Imagine a cascade of two integrators, being fed acceleration to produce velocity and height.

We start by creating a block that outputs a constant, the graviational acceleration

In [ ]:
gravity = bd.CONSTANT(g, name="gravity")

Next we add an integrator that will have acceleration as input and output velocity

In [ ]:
velocity = bd.INTEGRATOR(name="velocity", x0=0)  # initial height is initial velocity

Then we add another integrator that will have velocity as input and output height 

In [ ]:
height = bd.INTEGRATOR(name="height", x0=h0)

To see what's going on so we create a SCOPE block to plot the height and velocity.  We have lots of options here, we could plot height and velocity against each other for a phase plot, or plot them against time in separate scopes.

For this example we'll plot them both against time in the same scope plot.  We specify the line styles and where we want the legend.

In [ ]:
scope = bd.SCOPE(styles=["k", "r--"], loc="lower right")

Now it's time to connect the blocks together

In [ ]:
bd.connect(gravity, velocity)
bd.connect(velocity, height)
bd.connect(velocity, scope[0])
bd.connect(height, scope[1])

Next we compile the system and display a summary report

In [ ]:
bd.compile(verbose=True)
bd.report_summary()

Our system has sucessfull compiled.

The way to read this table is that height has 1 input which comes from the `velocity` block and is a NumPy array of shape (1,) and floating point.  `scope.0` has two inputs, one from `velocity` and one from `height`.

Now it's time to run our model.  We'll see what happens over the first 2 seconds.

In [ ]:
out = sim.run(bd, T=2)

Not surprisingly, the ball falls.  The height of ball decreases quadratically over time, while its velocity decreases linearly.  Our ball passes through the ground at around $t=1.4$s and keeps going into negative height territory -- that's because we haven't modeled the ground and what happens when the ball hits it.

We first need to do determine when the ball hits the ground, and then do something about it.  We'll tackle the second part first.  At impact we want the ball's velocity to change sign, to be bouncing upwards.  But we also want to scale it so that the ball doesn't bounce forever, and to model the energy loss (as heat) during the impact. This  scale factor is called the *coefficient of restitution*. It is 1 for a lossless bounce, and 0 for a lump of putty.

At impact we will modify the state of the first integrator referenced by the variable `velocity`

In [ ]:
def impact(event_block, state_map):
    state_map[velocity] *= -e  # reverse velocity and apply restitution

where `state_map` is a dictionary that maps a block reference to a view of the centralized state vector.

Next we need to determine when to invoke the `impact` function.  It must be called when the ball hits the ground, that is, when its height crosses from being positive to being negative. 

In [ ]:

ground = bd.EVENT("-", impact)


We do this with an EVENT block that invokes a specified function when its input signal crosses zero.  In this case, by providing the "-" argument (a negative going transition) the `impact` function will be called when the height changes from positive to negative.  The integrator normally takes quite large steps when integrating such simple dynamics, but it essentially performs a root search to hone in on the exact moment of crossing.

Now we need to wire this block into our system.

In [ ]:
bd.connect(height, ground)

and then incrementally recompile our system

In [ ]:
bd.compile()

Now we can run the simulation again and see the bouncing effect.

In [ ]:
out = sim.run(bd, T=10)